# Random Forest (supervised)
Uses labels during training. Shows what's achievable with labeled incident data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import statsmodels.api as sm

## Load data and split

In [ ]:
data = pd.read_csv('../data/labeled.csv', parse_dates=['timestamp'])
split_point = int(len(data) * 0.8)
train_data = data.iloc[:split_point]
test_data = data.iloc[split_point:]

feature_columns = ['memory_pct', 'roll_mean_1h', 'roll_std_1h', 'roll_mean_24h',
                   'diff_1', 'diff_6', 'hour', 'minute', 'time_of_day', 'day_of_week', 'is_weekend',
                   'dev_from_hour', 'zscore_hour', 'zscore_1h', 'zscore_24h',
                   'diff_24', 'diff_48', 'diff_96']

X_train = train_data[feature_columns].values
X_test = test_data[feature_columns].values
y_train = train_data['label'].values
y_test = test_data['label'].values

print('Anomalies in test:', y_test.sum())

## Train Random Forest

In [ ]:
random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest_model.fit(X_train, y_train)

forest_predictions = random_forest_model.predict(X_test)
forest_scores = random_forest_model.predict_proba(X_test)[:, 1]

print('Flagged points:', forest_predictions.sum())

## Plot detections

In [ ]:
timestamps = test_data['timestamp'].values
memory = test_data['memory_pct'].values

plt.figure(figsize=(14, 4))
plt.plot(timestamps, memory, color='lightblue')
plt.scatter(timestamps[y_test == 1], memory[y_test == 1], color='red', marker='x', label='True anomaly')
plt.scatter(timestamps[forest_predictions == 1], memory[forest_predictions == 1], color='green', s=10, label='RF flagged')
plt.title('Random Forest')
plt.ylabel('Memory %')
plt.legend()
plt.show()

## Evaluation

In [ ]:
print(classification_report(y_test, forest_predictions, target_names=['Normal','Anomaly'], zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, forest_predictions))
print('ROC-AUC:', round(roc_auc_score(y_test, forest_scores), 4))

## Feature importance

In [ ]:
feature_importance = pd.Series(random_forest_model.feature_importances_, index=feature_columns).sort_values()

plt.figure(figsize=(8, 6))
plt.barh(feature_importance.index, feature_importance.values)
plt.title('Feature Importance')
plt.show()

## Statistical analysis using statsmodels
Logistic regression to evaluate statistical significance of features.

In [ ]:
X_with_constant = sm.add_constant(X_train)
logit_model = sm.Logit(y_train, X_with_constant)
logit_result = logit_model.fit()
print(logit_result.summary())

In [ ]:
print('Pseudo R-squared:', round(logit_result.prsquared, 4))
print('LLR p-value:', logit_result.llr_pvalue)
print('AIC:', round(logit_result.aic, 2))
print('BIC:', round(logit_result.bic, 2))